## Setting

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import os
import json, csv, time, random
from datetime import datetime
from http.client import RemoteDisconnected
import httplib2
from collections import deque

httplib2.Http.timeout = 60

MANUAL_API_KEY = 'YOUR_YOUTUBE_API_KEY_HERE'  
API_KEY = MANUAL_API_KEY.strip()


def build_youtube_client():
    return build(
        'youtube',
        'v3',
        developerKey=API_KEY,
        http=httplib2.Http(timeout=60),
        cache_discovery=False,
    )

USE_BROAD_SEARCH = True

MAX_RETRIES     = 5
INITIAL_BACKOFF = 2
REQUEST_DELAY   = 0.5

REGION_CODE        = 'TW'
MAX_PER_PAGE       = 50
PAGES_PER_CATEGORY = 20

CATEGORY_IDS = {
    '1':  'Film & Animation',
    '2':  'Autos & Vehicles',
    '10': 'Music',
    '15': 'Pets & Animals',
    '17': 'Sports',
    '19': 'Travel & Events',
    '20': 'Gaming',
    '22': 'People & Blogs',
    '23': 'Comedy',
    '24': 'Entertainment',
    '25': 'News & Politics',
    '26': 'Howto & Style',
    '27': 'Education',
    '28': 'Science & Technology',
}

# --- BFS expansion settings ---
SEED_QUERY           = 'keyword'   # query used to fetch seed videos
SEED_VIDEO_COUNT     = 50      # how many seed videos to collect before BFS expansion
MAX_BFS_DEPTH        = 2       # how many hops to expand from each seed video
MAX_VIDEOS_PER_CHANNEL = 10    # how many additional videos to pull per channel at each hop
MAX_TOTAL_VIDEOS     = 2000    # overall cap on videos collected via BFS


## Utility functions

In [63]:
def retry_with_backoff(func):
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(REQUEST_DELAY)
            return func()
        except (RemoteDisconnected, TimeoutError, ConnectionResetError, BrokenPipeError) as e:
            if attempt < MAX_RETRIES - 1:
                wait = INITIAL_BACKOFF * (2 ** attempt) + random.uniform(0, 1)
                print(f'  Connection error: {e}, retrying in {wait:.1f}s ({attempt+1}/{MAX_RETRIES})')
                time.sleep(wait)
            else:
                raise
        except HttpError as e:
            if e.resp.status in [429, 500, 503]:
                if attempt < MAX_RETRIES - 1:
                    wait = INITIAL_BACKOFF * (2 ** attempt) + random.uniform(0, 1)
                    print(f'  HTTP {e.resp.status}, retrying in {wait:.1f}s ({attempt+1}/{MAX_RETRIES})')
                    time.sleep(wait)
                else:
                    raise
            else:
                raise

## Seed videos & BFS expansion

In [ ]:
def get_seed_videos(query=SEED_QUERY, count=SEED_VIDEO_COUNT):
    """Use the YouTube Data API search endpoint to collect an initial set of seed videos."""
    youtube = build_youtube_client()
    seed_videos = []
    next_page_token = None

    while len(seed_videos) < count:
        def _fetch():
            return youtube.search().list(
                part='snippet',
                q=query,
                maxResults=min(MAX_PER_PAGE, count - len(seed_videos)),
                type='video',
                regionCode=REGION_CODE,
                relevanceLanguage='zh-Hant',
                order='viewCount',
                videoDefinition='high',
                pageToken=next_page_token
            ).execute()

        response = retry_with_backoff(_fetch)
        items = response.get('items', [])
        print(f'  Seed page: {len(items)} videos')

        for item in items:
            video_id = item.get('id', {}).get('videoId')
            channel_id = item.get('snippet', {}).get('channelId')
            if not video_id or not channel_id:
                continue
            seed_videos.append({'video_id': video_id, 'channel_id': channel_id})

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            print('  No more pages')
            break

        time.sleep(1.0)

    return seed_videos[:count]


def get_uploads_playlist_id(channel_id):
    """Look up a channel's 'uploads' playlist ID, used to expand BFS to that channel's other videos."""
    youtube = build_youtube_client()

    def _fetch():
        return youtube.channels().list(
            part='contentDetails',
            id=channel_id
        ).execute()

    response = retry_with_backoff(_fetch)
    items = response.get('items', [])
    if not items:
        return None
    return items[0]['contentDetails']['relatedPlaylists']['uploads']


def get_videos_from_playlist(playlist_id, max_results=MAX_VIDEOS_PER_CHANNEL):
    """Pull up to max_results video IDs from a channel's uploads playlist."""
    youtube = build_youtube_client()
    video_ids = []
    next_page_token = None

    while len(video_ids) < max_results:
        def _fetch():
            return youtube.playlistItems().list(
                part='contentDetails',
                playlistId=playlist_id,
                maxResults=min(50, max_results - len(video_ids)),
                pageToken=next_page_token
            ).execute()

        try:
            response = retry_with_backoff(_fetch)
        except HttpError as e:
            print(f'  Playlist failed {playlist_id}: {e}')
            break

        items = response.get('items', [])
        for item in items:
            vid = item.get('contentDetails', {}).get('videoId')
            if vid:
                video_ids.append(vid)

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break

        time.sleep(1.0)

    return video_ids[:max_results]


def bfs_expand_from_seeds(seed_video_ids, max_depth=MAX_BFS_DEPTH,
                           max_videos_per_channel=MAX_VIDEOS_PER_CHANNEL,
                           max_total_videos=MAX_TOTAL_VIDEOS):
    """
    Breadth-first expansion starting from seed videos.
    At each hop: video -> its channel -> more videos from that channel's uploads -> ...
    Depth-limited and deduplicated by video_id / channel_id.
    """
    visited_videos = set()
    visited_channels = set()
    queue = deque([(vid, 0) for vid in seed_video_ids])
    expanded_video_ids = []

    while queue and len(expanded_video_ids) < max_total_videos:
        video_id, depth = queue.popleft()
        if video_id in visited_videos:
            continue
        visited_videos.add(video_id)
        expanded_video_ids.append(video_id)
        print(f'  BFS depth {depth}: {video_id} ({len(expanded_video_ids)}/{max_total_videos})')

        if depth >= max_depth:
            continue

        video_data = get_video_raw(video_id)
        if not video_data:
            continue
        channel_id = video_data.get('channel_id')
        if not channel_id or channel_id in visited_channels:
            continue
        visited_channels.add(channel_id)

        playlist_id = get_uploads_playlist_id(channel_id)
        if not playlist_id:
            continue

        neighbor_video_ids = get_videos_from_playlist(playlist_id, max_results=max_videos_per_channel)
        for nvid in neighbor_video_ids:
            if nvid not in visited_videos:
                queue.append((nvid, depth + 1))

        time.sleep(1.0)

    return expanded_video_ids

## Raw video data

In [65]:
def get_video_raw(video_id):
    """Fetch raw video data (no computation applied)"""
    youtube = build_youtube_client()

    def _fetch():
        return youtube.videos().list(
            part='snippet,statistics,contentDetails,topicDetails',
            id=video_id
        ).execute()

    try:
        response = retry_with_backoff(_fetch)
        if not (response and response['items']):
            return None

        v       = response['items'][0]
        stats   = v.get('statistics', {})
        content = v.get('contentDetails', {})
        topics  = v.get('topicDetails', {})

        return {
            'video_id':          video_id,
            'title':             v['snippet']['title'],
            'description':       v['snippet'].get('description') or None,
            'channel_id':        v['snippet']['channelId'],
            'channel_name':      v['snippet']['channelTitle'],
            'published_at':      v['snippet']['publishedAt'],
            'view_count':        int(stats.get('viewCount', 0)),
            'like_count':        int(stats.get('likeCount', 0)),
            'comment_count':     int(stats.get('commentCount', 0)),
            'duration':          content.get('duration', 'PT0S'),
            'topic_categories':  topics.get('topicCategories', []),  # raw Wikipedia URLs
        }
    except Exception as e:
        print(f'  Video failed {video_id}: {e}')
        return None

## Raw channel data

In [66]:
def get_channel_raw(channel_id):
    """Fetch raw channel data (no computation applied)"""
    youtube = build_youtube_client()

    def _fetch():
        return youtube.channels().list(
            part='snippet,statistics,topicDetails',
            id=channel_id
        ).execute()

    try:
        response = retry_with_backoff(_fetch)
        if not response['items']:
            return None

        ch    = response['items'][0]
        stats = ch.get('statistics', {})
        snip  = ch['snippet']

        return {
            'channel_id':        channel_id,
            'channel_name':      snip['title'],
            'custom_url':        snip.get('customUrl', None),
            'country':           snip.get('country', None),
            'created_at':        snip['publishedAt'],
            'subscriber_count':  int(stats.get('subscriberCount', 0)),
            'view_count':        int(stats.get('viewCount', 0)),
            'video_count':       int(stats.get('videoCount', 0)),
            'topic_categories':  ch.get('topicDetails', {}).get('topicCategories', []),
        }
    except Exception as e:
        print(f'  Channel failed {channel_id}: {e}')
        return None

## Save functions

In [67]:
def save_json(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f'Saved: {filename} ({len(data)} records)')


def save_csv(data, filename):
    if not data:
        return
    rows = []
    for item in data:
        row = {
            k: (json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v)
            for k, v in item.items()
        }
        rows.append(row)

    fieldnames = list(rows[0].keys())
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f'Saved: {filename} ({len(rows)} records)')

## Main Code

In [ ]:
if __name__ == '__main__':
    timestamp     = datetime.now().strftime('%Y%m%d_%H%M%S')
    all_videos    = []
    all_channels  = {}

    print('\n[Step 1] Collecting seed videos via YouTube Data API')
    print('=' * 60)
    seeds = get_seed_videos()
    seed_video_ids = [s['video_id'] for s in seeds]
    print(f'Collected {len(seed_video_ids)} seed videos')

    print('\n[Step 2] Expanding dataset via BFS from seed videos')
    print('=' * 60)
    expanded_video_ids = bfs_expand_from_seeds(seed_video_ids)
    print(f'BFS expansion produced {len(expanded_video_ids)} total videos')

    print('\n[Step 3] Fetching full video & channel data')
    print('=' * 60)
    for i, vid in enumerate(expanded_video_ids, 1):
        print(f'  [{i}/{len(expanded_video_ids)}] video_id: {vid}')

        video_data = get_video_raw(vid)
        if not video_data:
            continue

        video_data['category_id']   = None
        video_data['category_name'] = 'BFS Expansion'
        all_videos.append(video_data)

        ch_id = video_data['channel_id']
        if ch_id not in all_channels:
            print(f'    \u2192 Fetching channel: {video_data["channel_name"]}')
            ch_data = get_channel_raw(ch_id)
            if ch_data:
                all_channels[ch_id] = ch_data

        time.sleep(1.5)

    print(f'\nDone! {len(all_videos)} videos, {len(all_channels)} channels')

    channel_map = {ch['channel_id']: ch for ch in all_channels.values() if ch.get('channel_id')}
    merged_records = []
    for v in all_videos:
        ch = channel_map.get(v.get('channel_id'), {})
        merged = dict(v)
        merged.update({
            'channel_custom_url': ch.get('custom_url'),
            'channel_country': ch.get('country'),
            'channel_created_at': ch.get('created_at'),
            'channel_subscriber_count': ch.get('subscriber_count'),
            'channel_view_count': ch.get('view_count'),
            'channel_video_count': ch.get('video_count'),
            'channel_topic_categories': ch.get('topic_categories', []),
        })
        merged_records.append(merged)

    merged_file = f'merged_final_brands_{timestamp}.json'
    save_json(merged_records, merged_file)
    print(f'Merge complete: {merged_file} ({len(merged_records)} records)')